# Parkinson's Disease Detection using Machine Learning

## Overview

This project predicts Parkinson's Disease using voice biomarkers extracted from the UCI Parkinson's Dataset.

The notebook demonstrates:

- Exploratory Data Analysis
- Feature Engineering
- Data Balancing using SMOTE
- Cross Validation
- Multiple Machine Learning Models
- Threshold Optimization
- Experimental Voice Prediction
- Spiral Drawing Classification
- Gradio Web Interface

Author: Anurag Bisht

In [ ]:
# Imports & Dataset Download

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, recall_score,
                             precision_score, f1_score)
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head()

: 

In [ ]:

# Exploratory Data Analysis

print("Dataset Info")
print(df.info())

print("\n Missing Values")
print(df.isnull().sum())

print("\n Class Distribution ")
print(df['status'].value_counts())
print(f"\nParkinson's: {df['status'].sum()} samples")
print(f"Healthy: {(df['status'] == 0).sum()} samples")

# Class distribution plot
plt.figure(figsize=(6, 4))
sns.countplot(x='status', data=df, palette='Set2')
plt.title("Class Distribution (0=Healthy, 1=Parkinson's)")
plt.xticks([0, 1], ['Healthy', "Parkinson's"])
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(df.drop(columns=['name']).corr(), cmap='coolwarm', annot=False, linewidths=0.5)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()



In [ ]:
# Distribution of key voice features
key_features = ['MDVP:Fo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Shimmer', 'HNR', 'RPDE', 'PPE']
df[key_features + ['status']].groupby('status').mean().T.plot(kind='bar', figsize=(12, 5), colormap='Set1')
plt.title("Mean Feature Values by Class")
plt.ylabel("Mean Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Drop name column (not a feature)
df = df.drop(columns=['name'])

# Separate features and target
X = df.drop(columns=['status'])
y = df['status']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Train-test split BEFORE any oversampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaling done!")

In [ ]:
# Method 1: SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("=== After SMOTE ===")
print(f"Shape: {X_train_smote.shape}")
print(pd.Series(y_train_smote).value_counts())

# Method 2: SMOTETomek (SMOTE + cleaning borderline samples)
smotetomek = SMOTETomek(random_state=42)
X_train_st, y_train_st = smotetomek.fit_resample(X_train_scaled, y_train)

print("\n=== After SMOTETomek ===")
print(f"Shape: {X_train_st.shape}")
print(pd.Series(y_train_st).value_counts())

# Visualize class balance after oversampling
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(['Healthy', "Parkinson's"], y_train.value_counts().sort_index(), color=['#2ecc71', '#e74c3c'])
axes[0].set_title("Original Training Data")
axes[0].set_ylabel("Count")

axes[1].bar(['Healthy', "Parkinson's"], pd.Series(y_train_smote).value_counts().sort_index(), color=['#2ecc71', '#e74c3c'])
axes[1].set_title("After SMOTE")

axes[2].bar(['Healthy', "Parkinson's"], pd.Series(y_train_st).value_counts().sort_index(), color=['#2ecc71', '#e74c3c'])
axes[2].set_title("After SMOTETomek")

plt.tight_layout()
plt.show()


## Cross-Validation

Because this dataset is small, a single train/test split can give unstable results. The following section uses stratified 5-fold cross-validation with scaling and SMOTE inside each fold, which gives a more reliable estimate of baseline model performance.


In [ ]:

# Cross-validation for more reliable evaluation

from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
}

scoring = {
    'accuracy': 'accuracy',
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = []

for model_name, model in cv_models.items():
    pipeline = ImbPipeline(steps=[
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    scores = cross_validate(
        pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False
    )

    cv_results.append({
        'Model': model_name,
        'Accuracy Mean': scores['test_accuracy'].mean(),
        'Accuracy Std': scores['test_accuracy'].std(),
        'Recall Mean': scores['test_recall'].mean(),
        'Recall Std': scores['test_recall'].std(),
        'F1 Mean': scores['test_f1'].mean(),
        'F1 Std': scores['test_f1'].std(),
        'ROC-AUC Mean': scores['test_roc_auc'].mean(),
        'ROC-AUC Std': scores['test_roc_auc'].std()
    })

cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df.sort_values('ROC-AUC Mean', ascending=False).round(4))

print("Cross-validation summary:")
print("Higher mean scores are better. Lower standard deviation means the result is more stable across folds.")


In [ ]:

# Define baseline models
models = {
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
}

# Store results
results = []

for model_name, model in models.items():
    for data_name, (X_tr, y_tr) in [('SMOTE', (X_train_smote, y_train_smote)),
                                    ('SMOTETomek', (X_train_st, y_train_st))]:
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]

        recall    = recall_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        f1        = f1_score(y_test, y_pred)
        roc_auc   = roc_auc_score(y_test, y_prob)

        results.append({
            'Model': model_name,
            'Sampling': data_name,
            'Recall': round(recall, 4),
            'Precision': round(precision, 4),
            'F1 Score': round(f1, 4),
            'ROC-AUC': round(roc_auc, 4)
        })

        print(f"\n{model_name} + {data_name}")
        print(classification_report(y_test, y_pred, target_names=['Healthy', "Parkinson's"]))

results_df = pd.DataFrame(results)
print("\n=== Baseline Model Comparison Summary ===")
print(results_df.sort_values('Recall', ascending=False).to_string(index=False))
print("\nNote: these are baseline models without full hyperparameter tuning.")


In [ ]:
# Validation-based threshold selection
X_train_main, X_val, y_train_main, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.25, random_state=42, stratify=y_train
)

smote_for_threshold = SMOTE(random_state=42)
X_train_main_smote, y_train_main_smote = smote_for_threshold.fit_resample(X_train_main, y_train_main)

threshold_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
threshold_model.fit(X_train_main_smote, y_train_main_smote)

y_val_prob = threshold_model.predict_proba(X_val)[:, 1]

print("=== Validation Threshold Analysis ===")
print(f"{'Threshold':<12} {'Recall':<10} {'Precision':<12} {'F1':<10}")
print("-" * 45)

threshold_rows = []
for threshold in np.arange(0.1, 0.9, 0.05):
    y_val_pred = (y_val_prob >= threshold).astype(int)
    rec = recall_score(y_val, y_val_pred)
    pre = precision_score(y_val, y_val_pred, zero_division=0)
    f1  = f1_score(y_val, y_val_pred, zero_division=0)
    threshold_rows.append({'threshold': threshold, 'recall': rec, 'precision': pre, 'f1': f1})
    print(f"{threshold:<12.2f} {rec:<10.4f} {pre:<12.4f} {f1:<10.4f}")

thresholds_df = pd.DataFrame(threshold_rows)
best_row = thresholds_df.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
BEST_THRESHOLD = float(best_row['threshold'])

print(f"\nSelected threshold from validation set: {BEST_THRESHOLD:.2f}")

# Retrain final model on the full training set after threshold selection.
best_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
best_model.fit(X_train_smote, y_train_smote)

y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

# ROC Curve on the untouched test set
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'Test ROC Curve (AUC = {roc_auc_score(y_test, y_prob):.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve - XGBoost Final Model')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Final Evaluation on Untouched Test Set

y_pred_final = (y_prob >= BEST_THRESHOLD).astype(int)

print(f"=== Final Model Performance (XGBoost, Threshold={BEST_THRESHOLD:.2f}) ===\n")
print(classification_report(y_test, y_pred_final, target_names=['Healthy', "Parkinson's"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Healthy', "Parkinson's"],
            yticklabels=['Healthy', "Parkinson's"])
plt.title(f'Confusion Matrix - XGBoost (Threshold={BEST_THRESHOLD:.2f})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns)
feat_imp.sort_values(ascending=True).tail(15).plot(kind='barh', color='steelblue')
plt.title('Top 15 Important Features - XGBoost')
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()


In [ ]:
# Save Model & Scaler

# Save model
joblib.dump(best_model, 'parkinsons_xgboost_model.pkl')

# Save scaler
joblib.dump(scaler, 'parkinsons_scaler.pkl')

# Save threshold
joblib.dump(BEST_THRESHOLD, 'parkinsons_threshold.pkl')

print("Model saved as: parkinsons_xgboost_model.pkl")
print("Scaler saved as: parkinsons_scaler.pkl")
print("Threshold saved as: parkinsons_threshold.pkl")
print("\nAll files saved successfully!")

In [ ]:

# Experimental feature extraction from .wav using Parselmouth
import parselmouth
from parselmouth.praat import call

# The UCI dataset includes advanced nonlinear voice features that are not
# reliably extracted here. For the real-time demo, we use only features that
# this notebook extracts directly from the .wav file.
REALTIME_FEATURES = [
    'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)',
    'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP',
    'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
    'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR'
]

def extract_realtime_features(audio_path):
    """
    Extract deterministic voice features from a .wav file for the real-time demo.

    This does not compute RPDE, DFA, spread1, spread2, D2, or PPE, so the
    real-time model below is trained only on the feature subset available here.
    """
    sound = parselmouth.Sound(audio_path)

    pitch = call(sound, "To Pitch", 0.0, 75, 600)
    point_process = call(sound, "To PointProcess (periodic, cc)", 75, 600)

    mdvp_fo  = call(pitch, "Get mean", 0, 0, "Hertz")
    mdvp_fhi = call(pitch, "Get maximum", 0, 0, "Hertz", "Parabolic")
    mdvp_flo = call(pitch, "Get minimum", 0, 0, "Hertz", "Parabolic")

    mdvp_jitter_percent = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    mdvp_jitter_abs     = call(point_process, "Get jitter (local, absolute)", 0, 0, 0.0001, 0.02, 1.3)
    mdvp_rap            = call(point_process, "Get jitter (rap)", 0, 0, 0.0001, 0.02, 1.3)
    mdvp_ppq            = call(point_process, "Get jitter (ppq5)", 0, 0, 0.0001, 0.02, 1.3)
    jitter_ddp          = mdvp_rap * 3

    mdvp_shimmer  = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    shimmer_apq3  = call([sound, point_process], "Get shimmer (apq3)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    shimmer_apq5  = call([sound, point_process], "Get shimmer (apq5)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    shimmer_apq11 = call([sound, point_process], "Get shimmer (apq11)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    shimmer_dda   = shimmer_apq3 * 3

    # Praat may not expose shimmer dB for this object combination, so this is
    # a deterministic approximation rather than a random placeholder.
    mdvp_shimmer_db = -20 * np.log10(max(1 - mdvp_shimmer, 1e-6))

    harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
    hnr = call(harmonicity, "Get mean", 0, 0)
    nhr = 1 / (10 ** (hnr / 10)) if hnr > 0 else 0

    return {
        'MDVP:Fo(Hz)': mdvp_fo,
        'MDVP:Fhi(Hz)': mdvp_fhi,
        'MDVP:Flo(Hz)': mdvp_flo,
        'MDVP:Jitter(%)': mdvp_jitter_percent,
        'MDVP:Jitter(Abs)': mdvp_jitter_abs,
        'MDVP:RAP': mdvp_rap,
        'MDVP:PPQ': mdvp_ppq,
        'Jitter:DDP': jitter_ddp,
        'MDVP:Shimmer': mdvp_shimmer,
        'MDVP:Shimmer(dB)': mdvp_shimmer_db,
        'Shimmer:APQ3': shimmer_apq3,
        'Shimmer:APQ5': shimmer_apq5,
        'MDVP:APQ': shimmer_apq11,
        'Shimmer:DDA': shimmer_dda,
        'NHR': nhr,
        'HNR': hnr
    }

print("Real-time feature extraction function defined.")
print(f"Real-time demo features: {len(REALTIME_FEATURES)}")


In [ ]:

# Experimental Real-Time Prediction Pipeline

import sounddevice as sd
from scipy.io.wavfile import write as wav_write
import tempfile
import os

# Train a separate real-time demo model using only features that can be
# extracted from microphone audio in this notebook.
X_rt = df[REALTIME_FEATURES]
y_rt = df['status']

X_rt_train, X_rt_test, y_rt_train, y_rt_test = train_test_split(
    X_rt, y_rt, test_size=0.2, random_state=42, stratify=y_rt
)

realtime_scaler = StandardScaler()
X_rt_train_scaled = realtime_scaler.fit_transform(X_rt_train)
X_rt_test_scaled = realtime_scaler.transform(X_rt_test)

X_rt_train_smote, y_rt_train_smote = SMOTE(random_state=42).fit_resample(X_rt_train_scaled, y_rt_train)

realtime_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
realtime_model.fit(X_rt_train_smote, y_rt_train_smote)

y_rt_prob = realtime_model.predict_proba(X_rt_test_scaled)[:, 1]
y_rt_pred = (y_rt_prob >= 0.5).astype(int)

print("=== Real-time Demo Model Evaluation ===")
print(classification_report(y_rt_test, y_rt_pred, target_names=['Healthy', "Parkinson's"]))
print(f"ROC-AUC Score: {roc_auc_score(y_rt_test, y_rt_prob):.4f}")
print("\nImportant: this is an experimental demo model, not a medical diagnostic tool.")

REALTIME_THRESHOLD = 0.5

def predict_from_wav(audio_path):
    """Predict Parkinson's-like voice pattern from a .wav file path."""
    print(f"\nProcessing: {audio_path}")

    features = extract_realtime_features(audio_path)
    features_df = pd.DataFrame([features])[REALTIME_FEATURES]

    features_scaled = realtime_scaler.transform(features_df)
    prob = realtime_model.predict_proba(features_scaled)[:, 1][0]
    prediction = 1 if prob >= REALTIME_THRESHOLD else 0

    print("\n=== Experimental Prediction Result ===")
    print(f"Parkinson's-like voice pattern probability: {prob:.4f} ({prob*100:.1f}%)")
    print(f"Threshold used: {REALTIME_THRESHOLD}")
    if prediction == 1:
        print("Result: High Parkinson's-like voice pattern")
    else:
        print("Result: Low Parkinson's-like voice pattern")
    print("\nThis result is only a prototype estimate and should not be used as a diagnosis.")
    print("=" * 30)

    return prediction, prob


def record_and_predict(duration=5, sample_rate=22050):
    """Record voice from mic and run the experimental prediction."""
    print(f"Recording for {duration} seconds...")
    print("Please say 'Ahhh' in a steady sustained voice.")
    print("Recording starts in 3 seconds...")

    import time
    time.sleep(3)

    print("Recording now...")
    recording = sd.rec(int(duration * sample_rate),
                       samplerate=sample_rate,
                       channels=1, dtype='float32')
    sd.wait()

    print("Recording complete.")

    temp_path = tempfile.mktemp(suffix='.wav')
    wav_write(temp_path, sample_rate, recording)

    try:
        prediction, prob = predict_from_wav(temp_path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    return prediction, prob

print("Experimental real-time prediction pipeline ready.")
print("\nUsage:")
print("  predict_from_wav('your_audio.wav')")
print("  record_and_predict()")


In [ ]:

# Live Microphone Recording Demo
record_and_predict(duration=5)



## Real-time Demo Limitation

The real-time voice prediction is experimental. The UCI dataset provides precomputed biomedical voice features, while raw microphone input requires feature extraction from audio. This notebook avoids random placeholder features by training a separate real-time demo model only on features that are extracted directly from the `.wav` file.

The output should be interpreted as a Parkinson's-like voice-pattern estimate, not as a clinical diagnosis.


In [ ]:

# Generate a synthetic clean voice-like .wav for testing


from scipy.io.wavfile import write as wav_write

def generate_synthetic_voice(filename="synthetic_voice.wav", duration=5, sample_rate=22050):
    """Generate a clean sustained vowel-like tone for testing the pipeline."""
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    f0 = 120.0

    signal = (
        0.6  * np.sin(2 * np.pi * f0 * t) +
        0.3  * np.sin(2 * np.pi * f0 * 2 * t) +
        0.15 * np.sin(2 * np.pi * f0 * 3 * t) +
        0.08 * np.sin(2 * np.pi * f0 * 4 * t) +
        0.04 * np.sin(2 * np.pi * f0 * 5 * t)
    )

    signal = signal / np.max(np.abs(signal))
    signal_int = (signal * 32767).astype(np.int16)

    wav_write(filename, sample_rate, signal_int)
    print(f"Synthetic voice saved as: {filename}")
    return filename

filename = generate_synthetic_voice()
predict_from_wav(filename)


In [ ]:

# Example for your own file:
 #predict_from_wav("male.wav")



## Spiral Drawing Test

This section adds a separate spiral-image model. It does not mix image data with the voice-feature model. The spiral model expects a local image dataset with folders or parent folder names containing labels such as `healthy`, `control`, `parkinson`, `parkinsons`, or `pd`.

Recommended structure:

```text
spiral_dataset/
  healthy/
    image1.png
    image2.png
  parkinson/
    image1.png
    image2.png
```

You can also use nested folders such as `spiral/training/healthy` and `spiral/training/parkinson`; the loader checks parent folder names.


In [ ]:

# Separate Spiral Drawing Model

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

SPIRAL_DATASET_CANDIDATES = [
    Path("spiral_dataset"),
    Path.cwd() / "spiral_dataset",
    Path.home() / "Downloads" / "spiral_dataset",
]
SPIRAL_DATASET_DIR = next((path for path in SPIRAL_DATASET_CANDIDATES if path.exists()), SPIRAL_DATASET_CANDIDATES[0])
print(f"Spiral dataset path: {SPIRAL_DATASET_DIR.resolve()}")
SPIRAL_IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

spiral_model = None
spiral_model_ready = False
spiral_feature_size = None

def _spiral_label_from_path(path):
    parts = [part.lower() for part in path.parts]
    if any(part in {"healthy", "control", "controls", "normal"} for part in parts):
        return 0
    if any(part in {"parkinson", "parkinsons", "pd", "patient", "patients"} for part in parts):
        return 1
    return None


def extract_spiral_features(image_path, image_size=128, pixel_size=32):

    try:
        from PIL import Image, ImageOps
    except ImportError as exc:
        raise ImportError("Pillow is required for spiral image processing. Install it with: pip install pillow") from exc

    image = Image.open(image_path).convert("L")
    image = ImageOps.autocontrast(image)
    image = image.resize((image_size, image_size))

    arr = np.asarray(image, dtype=np.float32) / 255.0

    # Most spiral sheets are dark ink on a light background. If an image has a
    # dark background, invert it so higher ink intensity still means more stroke.
    if arr.mean() < 0.5:
        arr = 1.0 - arr

    ink = 1.0 - arr
    binary = ink > 0.25

    coords = np.argwhere(binary)
    if coords.size == 0:
        stats = np.zeros(14, dtype=np.float32)
    else:
        y_coords = coords[:, 0]
        x_coords = coords[:, 1]
        center_y = y_coords.mean() / image_size
        center_x = x_coords.mean() / image_size
        spread_y = y_coords.std() / image_size
        spread_x = x_coords.std() / image_size
        height = (y_coords.max() - y_coords.min() + 1) / image_size
        width = (x_coords.max() - x_coords.min() + 1) / image_size
        aspect_ratio = width / max(height, 1e-6)

        distances = np.sqrt((x_coords - x_coords.mean()) ** 2 + (y_coords - y_coords.mean()) ** 2)

        stats = np.array([
            binary.mean(),
            ink.mean(),
            ink.std(),
            center_x,
            center_y,
            spread_x,
            spread_y,
            width,
            height,
            aspect_ratio,
            distances.mean() / image_size,
            distances.std() / image_size,
            np.percentile(ink, 75),
            np.percentile(ink, 90),
        ], dtype=np.float32)

    small = Image.fromarray((ink * 255).astype(np.uint8)).resize((pixel_size, pixel_size))
    pixel_features = np.asarray(small, dtype=np.float32).reshape(-1) / 255.0

    return np.concatenate([stats, pixel_features])


def load_spiral_dataset(dataset_dir=SPIRAL_DATASET_DIR):
    dataset_dir = Path(dataset_dir)
    image_paths = [
        path for path in dataset_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SPIRAL_IMAGE_EXTENSIONS
    ]

    features = []
    labels = []
    used_paths = []

    for path in image_paths:
        label = _spiral_label_from_path(path)
        if label is None:
            continue
        features.append(extract_spiral_features(path))
        labels.append(label)
        used_paths.append(path)

    if not features:
        return None, None, []

    return np.vstack(features), np.asarray(labels), used_paths


if SPIRAL_DATASET_DIR.exists():
    X_spiral, y_spiral, spiral_paths = load_spiral_dataset(SPIRAL_DATASET_DIR)

    if X_spiral is None or len(np.unique(y_spiral)) < 2:
        print("Spiral dataset found, but it needs labeled images for both healthy and parkinson classes.")
    else:
        spiral_feature_size = X_spiral.shape[1]
        print(f"Loaded {len(y_spiral)} spiral images.")
        print(pd.Series(y_spiral).map({0: "Healthy", 1: "Parkinson's"}).value_counts())

        X_sp_train, X_sp_test, y_sp_train, y_sp_test = train_test_split(
            X_spiral, y_spiral, test_size=0.2, random_state=42, stratify=y_spiral
        )

        spiral_model = RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        )
        spiral_model.fit(X_sp_train, y_sp_train)

        y_sp_pred = spiral_model.predict(X_sp_test)
        y_sp_prob = spiral_model.predict_proba(X_sp_test)[:, 1]

        print("\n=== Spiral Model Evaluation ===")
        print(classification_report(y_sp_test, y_sp_pred, target_names=["Healthy", "Parkinson's"]))
        print(f"ROC-AUC Score: {roc_auc_score(y_sp_test, y_sp_prob):.4f}")

        joblib.dump(spiral_model, "spiral_random_forest_model.pkl")
        spiral_model_ready = True
        print("Spiral model saved as: spiral_random_forest_model.pkl")
else:
    print("No spiral dataset folder found yet.")
    print("Create a folder named 'spiral_dataset' with healthy/parkinson image folders, then rerun this cell.")


def predict_spiral_from_image(image_path):
    """Predict Parkinson's-like drawing pattern from a spiral image."""
    if not spiral_model_ready or spiral_model is None:
        return None, None, "Spiral model is not trained yet. Add a labeled spiral_dataset folder and rerun the spiral model cell."

    features = extract_spiral_features(image_path).reshape(1, -1)
    prob = spiral_model.predict_proba(features)[:, 1][0]
    prediction = 1 if prob >= 0.5 else 0

    if prediction == 1:
        label = "High Parkinson's-like spiral pattern"
    else:
        label = "Low Parkinson's-like spiral pattern"

    message = (
        f"{label}\n"
        f"Probability: {prob:.4f} ({prob * 100:.1f}%)\n"
        "Prototype only: this is not a medical diagnosis."
    )
    return prediction, float(prob), message



## App Interface

The following Gradio interface lets you upload or record a `.wav` voice sample and run the experimental real-time model from a simple browser UI.


In [ ]:

# Gradio App Interface

try:
    import gradio as gr

    APP_CSS = """
    .gradio-container {
        max-width: 1180px !important;
        margin: auto !important;
        font-family: Inter, Segoe UI, Arial, sans-serif !important;
        background: #f6f8fb;
    }
    #hero {
        padding: 28px 30px;
        border-radius: 16px;
        background: linear-gradient(135deg, #17324d 0%, #246b63 55%, #f2b84b 100%);
        color: white;
        margin-bottom: 18px;
        box-shadow: 0 16px 38px rgba(23, 50, 77, 0.18);
    }
    #hero h1 {
        margin: 0 0 8px 0;
        font-size: 34px;
        line-height: 1.12;
        letter-spacing: 0;
    }
    #hero p {
        margin: 0;
        font-size: 15px;
        line-height: 1.55;
        max-width: 860px;
        opacity: 0.96;
    }
    .panel {
        border: 1px solid #dfe7ef !important;
        border-radius: 14px !important;
        background: white !important;
        box-shadow: 0 8px 22px rgba(25, 39, 52, 0.06);
        padding: 14px !important;
    }
    .panel-title {
        font-weight: 700;
        color: #17324d;
        margin: 0 0 6px 0;
        font-size: 17px;
    }
    .panel-subtitle {
        color: #5b6775;
        font-size: 13px;
        line-height: 1.45;
        margin: 0 0 12px 0;
    }
    #analyze-btn {
        min-height: 48px;
        font-size: 16px;
        font-weight: 700;
        border-radius: 12px;
        background: #176b5d !important;
        border: none !important;
    }
    .result-card {
        padding: 16px 18px;
        border-radius: 14px;
        background: white;
        border: 1px solid #dfe7ef;
        box-shadow: 0 8px 22px rgba(25, 39, 52, 0.06);
        min-height: 118px;
    }
    .result-card h3 {
        margin: 0 0 8px 0;
        color: #17324d;
        font-size: 16px;
    }
    .result-card .score {
        font-size: 28px;
        font-weight: 800;
        color: #176b5d;
        margin-bottom: 6px;
    }
    .result-card .label {
        color: #3f4a56;
        line-height: 1.35;
        font-size: 14px;
    }
    .status-low { color: #176b5d; font-weight: 800; }
    .status-medium { color: #b87913; font-weight: 800; }
    .status-high { color: #b73535; font-weight: 800; }
    .summary-box {
        padding: 18px 20px;
        border-radius: 14px;
        border: 1px solid #dfe7ef;
        background: #ffffff;
        box-shadow: 0 8px 22px rgba(25, 39, 52, 0.06);
        color: #25313d;
        line-height: 1.55;
    }
    .summary-box h2 {
        margin: 0 0 8px 0;
        color: #17324d;
        font-size: 20px;
    }
    .summary-box p {
        margin: 6px 0;
    }
    .disclaimer {
        padding: 12px 14px;
        border-radius: 12px;
        background: #fff8e6;
        border: 1px solid #f1d48a;
        color: #59440d;
        font-size: 13px;
        line-height: 1.45;
    }
    """

    def _risk_class(prob):
        if prob >= 0.65:
            return "status-high", "High"
        if prob >= 0.40:
            return "status-medium", "Medium"
        return "status-low", "Low"

    def _score_card(title, prob, label, unavailable=False):
        if unavailable:
            return f"""
            <div class='result-card'>
                <h3>{title}</h3>
                <div class='score'>--</div>
                <div class='label'>No input provided for this test.</div>
            </div>
            """

        css_class, level = _risk_class(prob)
        return f"""
        <div class='result-card'>
            <h3>{title}</h3>
            <div class='score'>{prob * 100:.1f}%</div>
            <div class='label'><span class='{css_class}'>{level}</span> - {label}</div>
        </div>
        """

    def gradio_multimodal_predict(audio_path, spiral_image_path):
        voice_prob = None
        spiral_prob = None
        voice_prediction = None
        spiral_prediction = None

        if audio_path is not None:
            voice_prediction, voice_prob = predict_from_wav(audio_path)

        if spiral_image_path is not None:
            spiral_prediction, spiral_prob, _ = predict_spiral_from_image(spiral_image_path)

        available_probs = [prob for prob in [voice_prob, spiral_prob] if prob is not None]
        if available_probs:
            combined_prob = float(np.mean(available_probs))
        else:
            combined_prob = None

        if voice_prob is None:
            voice_card = _score_card("Voice Test", 0.0, "", unavailable=True)
        else:
            voice_label = (
                "Parkinson's-like voice pattern detected"
                if voice_prediction == 1
                else "Low Parkinson's-like voice pattern"
            )
            voice_card = _score_card("Voice Test", voice_prob, voice_label)

        if spiral_prob is None:
            spiral_card = _score_card("Spiral Test", 0.0, "", unavailable=True)
        else:
            spiral_label = (
                "Parkinson's-like drawing pattern detected"
                if spiral_prediction == 1
                else "Low Parkinson's-like drawing pattern"
            )
            spiral_card = _score_card("Spiral Test", spiral_prob, spiral_label)

        if combined_prob is None:
            combined_card = _score_card("Combined Score", 0.0, "", unavailable=True)
            summary = """
            <div class='summary-box'>
                <h2>Analysis Summary</h2>
                <p>Please provide a voice sample, a spiral drawing, or both, then click Analyze.</p>
                <div class='disclaimer'>This prototype is for educational use only and is not a medical diagnosis tool.</div>
            </div>
            """
            return voice_card, spiral_card, combined_card, summary

        css_class, combined_level = _risk_class(combined_prob)
        combined_card = _score_card(
            "Combined Score",
            combined_prob,
            "Average of available voice and spiral model probabilities"
        )

        inputs_used = []
        if voice_prob is not None:
            inputs_used.append("voice")
        if spiral_prob is not None:
            inputs_used.append("spiral drawing")
        input_text = " and ".join(inputs_used)

        summary = f"""
        <div class='summary-box'>
            <h2>Analysis Summary</h2>
            <p>The system analyzed <b>{input_text}</b> input and produced a
            <span class='{css_class}'>{combined_level.lower()}</span> combined Parkinson's-like pattern score.</p>
            <p>The voice and spiral tests are shown separately so the result is easier to understand.</p>
            <div class='disclaimer'>This is an educational prototype. It should not be used as a medical diagnosis or a replacement for professional clinical evaluation.</div>
        </div>
        """

        return voice_card, spiral_card, combined_card, summary

    with gr.Blocks(
        title="Parkinson's Multimodal Prototype",
        theme=gr.themes.Soft(primary_hue="teal", neutral_hue="slate"),
        css=APP_CSS
    ) as demo:
        gr.HTML(
            """
            <section id='hero'>
                <h1>Parkinson's Multimodal Screening Prototype</h1>
                <p>Analyze a sustained voice sample and a spiral drawing image through separate machine learning models. The app reports voice, spiral, and combined Parkinson's-like pattern scores in a simple dashboard.</p>
            </section>
            """
        )

        with gr.Row(equal_height=True):
            with gr.Column(scale=1, elem_classes=["panel"]):
                gr.HTML(
                    """
                    <p class='panel-title'>Voice Test</p>
                    <p class='panel-subtitle'>Record or upload a clear 5-second sustained "Ahhh" voice sample.</p>
                    """
                )
                audio_input = gr.Audio(
                    sources=["upload", "microphone"],
                    type="filepath",
                    label="Voice sample"
                )

            with gr.Column(scale=1, elem_classes=["panel"]):
                gr.HTML(
                    """
                    <p class='panel-title'>Spiral Drawing Test</p>
                    <p class='panel-subtitle'>Upload a spiral drawing image for motor-pattern analysis.</p>
                    """
                )
                spiral_input = gr.Image(
                    sources=["upload"],
                    type="filepath",
                    label="Spiral drawing"
                )

        with gr.Row():
            predict_button = gr.Button(
                "Analyze Voice and Spiral",
                variant="primary",
                elem_id="analyze-btn"
            )

        with gr.Row(equal_height=True):
            voice_card = gr.HTML()
            spiral_card = gr.HTML()
            combined_card = gr.HTML()

        summary_box = gr.HTML(
            """
            <div class='summary-box'>
                <h2>Analysis Summary</h2>
                <p>Add a voice sample, a spiral image, or both, then click Analyze.</p>
                <div class='disclaimer'>This prototype is for educational use only and is not a medical diagnosis tool.</div>
            </div>
            """
        )

        predict_button.click(
            fn=gradio_multimodal_predict,
            inputs=[audio_input, spiral_input],
            outputs=[voice_card, spiral_card, combined_card, summary_box]
        )

    demo.launch()

except ImportError:
    print("Gradio is not installed in this notebook kernel.")
    print("Install it with:")
    print("import sys")
    print("!{sys.executable} -m pip install gradio")



## Conclusion

This notebook builds baseline machine learning classifiers for the UCI Parkinson's voice-feature dataset. Because the dataset is small, stratified cross-validation is included to provide a more stable estimate of model performance than a single split alone.

The notebook also includes an experimental real-time voice demo, a separate spiral drawing model, and a Gradio app interface that can show voice, spiral, and combined prototype scores. To avoid fake/random voice features, the voice demo uses a separate model trained only on features that can be extracted directly from audio in this notebook. Because the datasets are small and the audio/image pipelines are not clinically validated, the result should be treated as a prototype pattern estimate rather than a medical diagnosis.
